In [ ]:
import json
import sys
import pandas as pd
import numpy as np
import random
import re
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import unicodedata
import spacy
import numpy as np
from scipy.stats import chi2_contingency
from tenacity import retry, stop_after_attempt, wait_exponential
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from scipy.optimize import milp, LinearConstraint, Bounds
import emoji
from empath import Empath
from nrclex import NRCLex
from nrclex.core import DEFAULT_LEXICON_FILENAME

import anthropic
from openai import OpenAI

In [ ]:
OAI_KEY = 'INSERT KEY HERE'
ANTHROPIC_KEY = 'INSERT KEY HERE'

client = OpenAI(api_key=OAI_KEY)
client2 = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [ ]:
SEED = 67

In [ ]:
nlp = spacy.load("en_core_web_trf")

EMPATH_CATEGORIES = [
    'positive_emotion', 'joy', 'cheerfulness', 'contentment', 'affection', 'love',
    'optimism', 'pride', 'zest',
    'negative_emotion', 'sadness', 'disappointment', 'suffering', 'torment',
    'nervousness', 'fear', 'timidity',
    'anger', 'rage', 'aggression', 'irritability', 'exasperation', 'hate',
    'disgust', 'shame',
    'sympathy', 'emotional', 'warmth',
    'swearing_terms',
    
    'thinking', 'order', 'confusion', 'anticipation', 'deception',
]

NRC_CATEGORIES_KEEP = {'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise',
                       'positive', 'negative'}

empath_lex = Empath()
PSYCH_VOCAB = set()
for cat in EMPATH_CATEGORIES:
    PSYCH_VOCAB.update(w.lower() for w in empath_lex.cats.get(cat, []))

_nrc = NRCLex()
try:
    nrc_dict = _nrc.lexicon
except AttributeError:
    nrc_dict = _nrc.__lexicon__

for word, emotions in nrc_dict.items():
    if any(e in NRC_CATEGORIES_KEEP for e in emotions):
        PSYCH_VOCAB.add(word.lower())

print(f"PSYCH_VOCAB size: {len(PSYCH_VOCAB)}")

In [ ]:
#posts_text = pd.read_csv('POSTS_subset_text.csv', encoding='ISO-8859-1')

fullset_essays_text = pd.read_csv('ESSAYS_fullset_text.csv', encoding='ISO-8859-1')
subset_essays_text = pd.read_csv('ESSAYS_subset_text.csv', encoding='ISO-8859-1')

subset_facebook_text = pd.read_csv('FACEBOOK_subset_text.csv', encoding='ISO-8859-1')

#texts = [posts_text, fullset_essays_text, subset_essays_text, subset_facebook_text]
texts = [fullset_essays_text, subset_essays_text, subset_facebook_text['STATUS'].astype(str)]

In [ ]:
# Implements the lexical evidence-channel ablation described
# in the Content-Masking Ablation section.

#commented out sections only apply to the unavailable student introduction post data
def _normalize(text):
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[–—]', '-', text)
    text = re.sub(r"[\u2018\u2019\u02BC]", "'", text)
    text = re.sub(r'[\u201C\u201D]', '"', text)
    return text

LINE_START = r"^[\s\-\*•]*(?:\d+[\.\)]\s*)?"

'''
ALL_TEMPLATE_PATTERNS = [
    re.compile(
        LINE_START +
        r"what do you do when you'?re not in the [REDACTED]\??\s*"
        r"(?:\([^)]*\))?",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"what(?:'?s| is) (?:something|one thing) interesting about you\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"why are you taking "
        r"(?:cs\s?-?7637\s*:?\s*)?"
        r"([REDACTED]??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"what do you hope to (?:get|gain|learn) (?:out )?(?:of|from) this course\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what(?:'?s| is) your name\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"where do you live\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what other [REDACTED] courses have you taken(?:\s+so far)?\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what courses do you plan to take(?:\s+next semester)?\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what specialization are you planning\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
    r"#\s*conn?e+c?t+[\s_\-]*me\b",
    re.IGNORECASE
    ),
]
'''

REPLACEMENT = '_'

'''
def remove_template_questions(text):
    text = _normalize(text)
    for pattern in ALL_TEMPLATE_PATTERNS:
        text = pattern.sub(REPLACEMENT, text)
    text = re.sub(r'(\s*_\s*)+', ' _ ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text\
'''


#PROGRAM_TERMS = {[REDACTED]}

COURSE_NUM_PATTERN = re.compile(r'\b[A-Z]{2,4}\d{4}\b')

URL_PATTERNS = [
    re.compile(r'https?://[^\s<>"\'\[\]]+', re.IGNORECASE),
    re.compile(r'www\.[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:/[^\s<>"\'\[\]]*)?', re.IGNORECASE),
]

TEXT_EMOTICON_PATTERN = re.compile(
    r"""(?<![a-zA-Z])(?:
        [;:=][-']?[)(DPpOo3/\\|*]    |  # standard :) :D ;P etc.
        [)(DPp]['-]?[;:=]             |  # reversed (: D:
        <3                             |  # heart
        [=][\)(\\/]                    |  # =) =( =/
        >\.<                              # >.<
    )(?![a-zA-Z])""",
    re.VERBOSE
)

FUNCTION_POS = {
    'ADP',
    'AUX',
    'CCONJ',
    'DET',
    'PART',
    'PRON',
    'SCONJ',
    'INTJ',
    'PUNCT',
    'SPACE',
}

FUNCTION_WORD_WHITELIST = {

    'something', 'everything', 'nothing', 'anything',
    'someone', 'everyone', 'no one', 'anyone',
    'somebody', 'everybody', 'nobody', 'anybody',

    'lots', 'much', 'many', 'few', 'several', 'some', 'any',
    'more', 'most', 'less', 'least', 'enough',
    'very', 'really', 'quite', 'pretty', 'too', 'so',

    'not', "n't", 'never', 'no', 'neither', 'nor',

    'also', 'just', 'still', 'already', 'yet', 'even',
    'always', 'often', 'sometimes', 'usually',
    'here', 'there', 'where', 'when', 'how', 'why', 'what', 'who',
    'again', 'then', 'now', 'only', 'else',
    'between', 'beyond', 'ago',
}

REPLACEMENT = '_'


def extract_emoticons(text):
    """Find all emoticon positions so we can preserve them."""
    positions = []

    for m in TEXT_EMOTICON_PATTERN.finditer(text):
        positions.append((m.start(), m.end(), m.group()))

    for m in emoji.emoji_list(text):
        positions.append((m['match_start'], m['match_end'], m['emoji']))

    return sorted(positions, key=lambda x: x[0])


def strip_urls(text):
    for pat in URL_PATTERNS:
        text = pat.sub('', text)
    return text

'''
def replace_program_terms(text):
    text = COURSE_NUM_PATTERN.sub(REPLACEMENT, text)

    for term in sorted(PROGRAM_TERMS, key=len, reverse=True):
        text = re.sub(r'\b' + re.escape(term) + r'\b', REPLACEMENT, text)

    return text
'''

def filter_text(text):
    text = strip_urls(text)
    #text = remove_template_questions(text)
    #text = replace_program_terms(text)

    emoticon_positions = extract_emoticons(text)

    emoticon_ranges = set()
    for start, end, _ in emoticon_positions:
        for i in range(start, end):
            emoticon_ranges.add(i)

    doc = nlp(text)

    result = []
    for tok in doc:
        tok_chars = set(range(tok.idx, tok.idx + len(tok.text)))
        if tok_chars & emoticon_ranges:
            result.append(tok.text)
            continue

        if tok.pos_ in ('SPACE', 'PUNCT') or tok.is_space or tok.is_punct:
            result.append(tok.text)
            continue

        if tok.pos_ in FUNCTION_POS:
            result.append(tok.text)
            continue

        if tok.text.lower() in FUNCTION_WORD_WHITELIST:
            result.append(tok.text)
            continue

        if tok.lemma_.lower() in PSYCH_VOCAB or tok.text.lower() in PSYCH_VOCAB:
            result.append(tok.text)
            continue

        if tok.pos_ == 'NUM':
            result.append(REPLACEMENT)
            continue

        result.append(REPLACEMENT)

    output = ''
    for i, tok in enumerate(doc):
        output += result[i]
        if tok.whitespace_:
            output += tok.whitespace_

    output = re.sub(r'(_\s*)+', '_ ', output)

    lines = [line.strip() for line in output.split('\n')]
    lines = [line for line in lines if line and line != REPLACEMENT]
    output = '\n'.join(lines)

    return output

'''
ablated_posts = []
for text in tqdm(posts_text):
    ablated_posts.append(filter_text(posts_text))
'''

ablated_full_essay = []
for text in tqdm(fullset_essays_text.iloc[:, -1]
    .tolist()):
    ablated_full_essay.append(filter_text(text))


ablated_subset_essay = []
for text in tqdm(subset_essays_text.iloc[:, -1].to_list()):
    ablated_subset_essay.append(filter_text(text))


ablated_facebook = []
for text in tqdm(subset_facebook_text["STATUS"].astype(str).tolist()):
    ablated_facebook.append(filter_text(text))

ablated_full_essay = pd.DataFrame(
    ablated_full_essay,
    index=fullset_essays_text.index
)
ablated_subset_essay = pd.DataFrame(
    ablated_subset_essay,
    index=subset_essays_text.index
)
ablated_facebook = pd.DataFrame(
    ablated_facebook,
    index=subset_facebook_text.index
)

#ablated_texts = [ablated_posts, ablated_full_essay, ablated_subset_essay, ablated_facebook]
ablated_texts = [ablated_full_essay, ablated_subset_essay, ablated_facebook]

In [ ]:
original_basic = ("You are an expert in inferring Big-5 personality traits from text.","Analyze this text for personality traits. Classify each Big Five trait as either 'low' or 'high'. Return your results in the following JSON format without explanation:")
original_basic_ex = ("You are an expert in inferring Big-5 personality traits from text. Your justifications are always evidence-base and grounded in direct quotes from the text.", "Your task is to analyze the provided text, classify the author's Big Five personality traits, and provide a detailed justification for your analysis. Classify each Big Five trait as either 'low' or 'high', no deviation allowed. Return your detailed justification and results in the following JSON format:")

ling_basic = ("You are an expert in inferring Big-5 personality traits from linguistic patterns in text.","Analyze this text for personality traits based only on writing style, word choice, and language patterns. Classify each Big Five trait as either 'low' or 'high'. Return your results in the following JSON format without explanation: ")
ling_basic_ex = ("You are an expert in inferring Big-5 personality traits from linguistic patterns in text.","Analyze this text for personality traits based only on writing style, word choice, and language patterns. Classify each Big Five trait as either 'low' or 'high'. If the linguistic evidence is insufficient, make your best inference based on available patterns, but always provide a classification, and note this in your explanation. Return your explanation and results in the following JSON format: ")

prompts = [original_basic, original_basic_ex, ling_basic, ling_basic_ex]

In [ ]:
models = ["gpt-4o-mini-2024-07-18", "o3-mini-2025-01-31", "gpt-5.4-mini-2026-03-17", "gpt-4o-2024-11-20", "claude-haiku-4-5-20251001"]

In [ ]:
# Runs the model/prompt configurations outlined across the 
# Model Selection/Prompts/LLM Personality Trait Prediction/Content-Masking Ablation 
# sections of the Methodology

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=4, max=10)
)
def run_model(model, prompt, text):
    if model == "claude-haiku-4-5-20251001":
        if prompt in [original_basic, ling_basic]:
            tools = [
                {
                    "name": "record_personality_profile",
                    "description": "Records the Big Five personality traits for each classification.",
                    "strict": True,
                    "input_schema": {
                        "type": "object",
                        "properties": {
                            "Openness": {
                                "type": "object",
                                "description": "The analysis for the Openness trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    }
                                },
                                "required": ["classification"],
                                "additionalProperties": False
                            },
                            "Conscientiousness": {
                                "type": "object",
                                "description": "The analysis for the Conscientiousness trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    }
                                },
                                "required": ["classification"],
                                "additionalProperties": False
                            },
                            "Extroversion": {
                                "type": "object",
                                "description": "The analysis for the Extroversion trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    }
                                },
                                "required": ["classification"],
                                "additionalProperties": False
                            },
                            "Agreeableness": {
                                "type": "object",
                                "description": "The analysis for the Agreeableness trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    }
                                },
                                "required": ["classification"],
                                "additionalProperties": False
                            },
                            "Neuroticism": {
                                "type": "object",
                                "description": "The analysis for the Neuroticism trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    }
                                },
                                "required": ["classification"],
                                "additionalProperties": False
                            }
                        },
                        "required": ["Openness", "Conscientiousness", "Extroversion", "Agreeableness", "Neuroticism"],
                        "additionalProperties": False
                    }
                }
            ]

            expected_format = """
            {
              "Openness": {"classification": "low/high"},
              "Conscientiousness": {"classification": "low/high"},
              "Extroversion": {"classification": "low/high"},
              "Agreeableness": {"classification": "low/high"},
              "Neuroticism": {"classification": "low/high"}
            }
            """
        else:
            tools = [
                {
                    "name": "record_personality_profile",
                    "description": (
                        "Records the Big Five personality traits and a "
                        "justification for each classification."
                    ),
                    "strict": True,
                    "input_schema": {
                        "type": "object",
                        "properties": {
                            "Openness": {
                                "type": "object",
                                "description": "The analysis for the Openness trait.",
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    },
                                    "justification": {
                                        "type": "string",
                                        "description": (
                                            "Reasoning based on a direct quote from the text."
                                        )
                                    }
                                },
                                "required": [
                                    "classification",
                                    "justification"
                                ],
                                "additionalProperties": False
                            },
                            "Conscientiousness": {
                                "type": "object",
                                "description": (
                                    "The analysis for the Conscientiousness trait."
                                ),
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    },
                                    "justification": {
                                        "type": "string",
                                        "description": (
                                            "Reasoning based on a direct quote from the text."
                                        )
                                    }
                                },
                                "required": [
                                    "classification",
                                    "justification"
                                ],
                                "additionalProperties": False
                            },
                            "Extroversion": {
                                "type": "object",
                                "description": (
                                    "The analysis for the Extroversion trait."
                                ),
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    },
                                    "justification": {
                                        "type": "string",
                                        "description": (
                                            "Reasoning based on a direct quote from the text."
                                        )
                                    }
                                },
                                "required": [
                                    "classification",
                                    "justification"
                                ],
                                "additionalProperties": False
                            },
                            "Agreeableness": {
                                "type": "object",
                                "description": (
                                    "The analysis for the Agreeableness trait."
                                ),
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    },
                                    "justification": {
                                        "type": "string",
                                        "description": (
                                            "Reasoning based on a direct quote from the text."
                                        )
                                    }
                                },
                                "required": [
                                    "classification",
                                    "justification"
                                ],
                                "additionalProperties": False
                            },
                            "Neuroticism": {
                                "type": "object",
                                "description": (
                                    "The analysis for the Neuroticism trait."
                                ),
                                "properties": {
                                    "classification": {
                                        "type": "string",
                                        "enum": ["low", "high"]
                                    },
                                    "justification": {
                                        "type": "string",
                                        "description": (
                                            "Reasoning based on a direct quote from the text."
                                        )
                                    }
                                },
                                "required": [
                                    "classification",
                                    "justification"
                                ],
                                "additionalProperties": False
                            }
                        },
                        "required": [
                            "Openness",
                            "Conscientiousness",
                            "Extroversion",
                            "Agreeableness",
                            "Neuroticism"
                        ],
                        "additionalProperties": False
                    }
                }
            ]
            expected_format = """
                {
                  "Openness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Conscientiousness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Extroversion": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Agreeableness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Neuroticism": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  }
                }
                """

        role_pre_prompt = prompt[0]
        role_q_prompt = prompt[1] + expected_format + ". Text Sample: " + text

        response = client2.messages.create(
            model=model,
            max_tokens=3000,
            temperature=0.0,
            system=role_pre_prompt,
            tools=tools,
            tool_choice={"type": "tool", "name": "record_personality_profile"},
            messages=[
                {"role": "user", "content": role_q_prompt}
            ]
        )

        tool_use_block = None

        for block in response.content:
            if block.type == "tool_use":
                tool_use_block = block
                break
        
        if tool_use_block is None:
            raise ValueError(
                f"No tool_use block found. "
                f"Stop reason: {response.stop_reason}"
            )
        
        if response.stop_reason != "tool_use":
            raise ValueError(
                f"API call did not stop with 'tool_use'. "
                f"Reason: {response.stop_reason}"
            )
        
        json_arguments = json.dumps(
            tool_use_block.input,
            ensure_ascii=False
        )
        
        return json_arguments
        
    else:
        if prompt not in [original_basic, ling_basic]:
            tools = [
                {
                    "type": "function",
                    "function": {
                        "name": "record_personality_profile",
                        "description": "Records the Big Five personality traits and a justification for each classification.",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "Openness": {
                                    "type": "object",
                                    "description": "The analysis for the Openness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        },
                                        "justification": {
                                            "type": "string",
                                            "description": "Reasoning based on a direct quote from the text."
                                        }
                                    },
                                    "required": ["classification", "justification"]
                                },
                                "Conscientiousness": {
                                    "type": "object",
                                    "description": "The analysis for the Conscientiousness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        },
                                        "justification": {
                                            "type": "string",
                                            "description": "Reasoning based on a direct quote from the text."
                                        }
                                    },
                                    "required": ["classification", "justification"]
                                },
                                "Extroversion": {
                                    "type": "object",
                                    "description": "The analysis for the Extroversion trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        },
                                        "justification": {
                                            "type": "string",
                                            "description": "Reasoning based on a direct quote from the text."
                                        }
                                    },
                                    "required": ["classification", "justification"]
                                },
                                "Agreeableness": {
                                    "type": "object",
                                    "description": "The analysis for the Agreeableness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        },
                                        "justification": {
                                            "type": "string",
                                            "description": "Reasoning based on a direct quote from the text."
                                        }
                                    },
                                    "required": ["classification", "justification"]
                                },
                                "Neuroticism": {
                                    "type": "object",
                                    "description": "The analysis for the Neuroticism trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        },
                                        "justification": {
                                            "type": "string",
                                            "description": "Reasoning based on a direct quote from the text."
                                        }
                                    },
                                    "required": ["classification", "justification"]
                                }
                            },
                            "required": ["Openness", "Conscientiousness", "Extroversion", "Agreeableness", "Neuroticism"]
                        }
                    }
                }
            ]

            expected_format = """
                {
                  "Openness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Conscientiousness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Extroversion": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Agreeableness": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  },
                  "Neuroticism": {
                    "classification": "low/high",
                    "justification": "My reasoning is based on the quote: '[Direct quote from the text]'"
                  }
                }
                """
        else:
            tools = [
                {
                    "type": "function",
                    "function": {
                        "name": "record_personality_profile",
                        "description": "Records the Big Five personality traits for each classification.",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "Openness": {
                                    "type": "object",
                                    "description": "The analysis for the Openness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        }
                                    },
                                    "required": ["classification"]
                                },
                                "Conscientiousness": {
                                    "type": "object",
                                    "description": "The analysis for the Conscientiousness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        }
                                    },
                                    "required": ["classification"]
                                },
                                "Extroversion": {
                                    "type": "object",
                                    "description": "The analysis for the Extroversion trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        }
                                    },
                                    "required": ["classification"]
                                },
                                "Agreeableness": {
                                    "type": "object",
                                    "description": "The analysis for the Agreeableness trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        }
                                    },
                                    "required": ["classification"]
                                },
                                "Neuroticism": {
                                    "type": "object",
                                    "description": "The analysis for the Neuroticism trait.",
                                    "properties": {
                                        "classification": {
                                            "type": "string",
                                            "enum": ["low", "high"]
                                        }
                                    },
                                    "required": ["classification"]
                                }
                            },
                            "required": ["Openness", "Conscientiousness", "Extroversion", "Agreeableness", "Neuroticism"]
                        }
                    }
                }
            ]
            expected_format = """
            {
              "Openness": {
                "classification": "low/high"
              },
              "Conscientiousness": {
                "classification": "low/high"
              },
              "Extroversion": {
                "classification": "low/high"
              },
              "Agreeableness": {
                "classification": "low/high"
              },
              "Neuroticism": {
                "classification": "low/high"
              }
            }
            """


        role_pre_prompt = prompt[0]

        role_q_prompt = (
            prompt[1]
            + expected_format
            + ". Text Sample: "
            + text
        )

        if model in ["o3-mini-2025-01-31", "gpt-5.4-mini-2026-03-17"]:
            response = client.chat.completions.create(
                model=model,
                tools=tools,
                tool_choice={"type": "function", "function": {"name": "record_personality_profile"}},
                messages=[
                    {"role": "system", "content": role_pre_prompt},
                    {"role": "user", "content": role_q_prompt}
                ],
                temperature=1,
                max_completion_tokens=3000
            )
        else:
            response = client.chat.completions.create(
                model=model,
                tools=tools,
                tool_choice="required",
                messages=[
                    {"role": "system", "content": role_pre_prompt},
                    {"role": "user", "content": role_q_prompt}
                ],
                temperature=0.0,
                max_tokens=3000
            )

        tool_call = response.choices[0].message.tool_calls[0]
        json_arguments = tool_call.function.arguments

        finish_reason = response.choices[0].finish_reason

        if finish_reason != "tool_calls":
            raise ValueError(f"API call did not finish with 'tool_calls'. Reason: {finish_reason}")

        return json_arguments


def extract_personality(model, prompt, text):
    result = run_model(model, prompt, text)
    personality_dict = json.loads(result)
    return personality_dict

In [ ]:
OUTPUT_DIR = Path("personality_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CHECKPOINT_EVERY = 25

prompt_configs = [
    ("original_basic", original_basic),
    ("original_basic_ex", original_basic_ex),
    ("ling_basic", ling_basic),
    ("ling_basic_ex", ling_basic_ex),
]

model_configs = [
    ("gpt4o_mini", "gpt-4o-mini-2024-07-18"),
    ("o3_mini", "o3-mini-2025-01-31"),
    ("gpt54_mini", "gpt-5.4-mini-2026-03-17"),
    ("gpt4o", "gpt-4o-2024-11-20"),
    ("claude_haiku", "claude-haiku-4-5-20251001"),
]

'''
(
    "posts_text",
    posts_text.iloc[:, -1].astype(str),
),
'''
'''
(
    "ablated_posts",
    ablated_posts.iloc[:, -1].astype(str),
)
'''
corpus_configs = [
    (
        "essay_fullset",
        fullset_essays_text.iloc[:, -1].astype(str),
    ),
    (
        "essay_subset",
        subset_essays_text.iloc[:, -1].astype(str),
    ),
    (
        "facebook_subset",
        ablated_facebook.iloc[:, -1].astype(str),
    ),
    (
        "ablated_essay_fullset",
        ablated_full_essay.iloc[:, -1].astype(str),
    ),
    (
        "ablated_essay_subset",
        ablated_subset_essay.iloc[:, -1].astype(str),
    ),
    (
        "ablated_facebook_subset",
        ablated_facebook.astype(str),
    ),
]

ablated_texts = [ablated_full_essay, ablated_subset_essay, ablated_facebook]



TRAITS = [
    "Openness",
    "Conscientiousness",
    "Extroversion",
    "Agreeableness",
    "Neuroticism",
]

In [ ]:
#pre-ablation loop AND post-ablation loop (both are run at once)

# WARNING: this takes several hours to fully complete
CLASSIFICATION_MAP = {
    "low": 0,
    "high": 1,
}
def flatten_personality(result):
    flattened = {}

    for trait in TRAITS:
        trait_result = result.get(trait, {})

        if isinstance(trait_result, dict):
            classification = (
                trait_result
                .get("classification")
                .strip()
                .lower()
            )
            
            if classification not in CLASSIFICATION_MAP:
                raise ValueError(
                    f"Unexpected classification for {trait}: "
                    f"{classification!r}"
                )
            
            flattened[f"{trait}_classification"] = (
                CLASSIFICATION_MAP[classification]
            )
            flattened[f"{trait}_justification"] = (
                trait_result.get("justification")
            )
        else:
            flattened[f"{trait}_classification"] = trait_result
            flattened[f"{trait}_justification"] = None

    return flattened

for corpus_name, corpus_texts in corpus_configs:
    for prompt_name, prompt in prompt_configs:
        for model_name, model in model_configs:

            config_name = (
                f"{prompt_name}_{model_name}_{corpus_name}"
            )

            output_path = OUTPUT_DIR / f"{config_name}.csv"

            if output_path.exists():
                existing = pd.read_csv(output_path)

                if "status" in existing.columns:
                    successful = existing[
                        existing["status"] == "ok"
                    ].copy()
                else:
                    successful = existing.copy()

                results = successful.to_dict("records")

                completed_indices = set(
                    successful["source_index"]
                    .astype(str)
                    .tolist()
                )
            else:
                results = []
                completed_indices = set()

            unsaved_count = 0

            progress = tqdm(
                corpus_texts.items(),
                total=len(corpus_texts),
                desc=config_name,
            )

            for source_index, text in progress:
                source_key = str(source_index)

                if source_key in completed_indices:
                    continue

                try:
                    personality = extract_personality(
                        model,
                        prompt,
                        text,
                    )

                    row = {
                        "source_index": source_index,
                        "status": "ok",
                        "error": None,
                    }

                    row.update(
                        flatten_personality(personality)
                    )

                    completed_indices.add(source_key)

                except Exception as error:
                    row = {
                        "source_index": source_index,
                        "status": "error",
                        "error": (
                            f"{type(error).__name__}: {error}"
                        ),
                    }

                results.append(row)
                unsaved_count += 1

                if unsaved_count >= CHECKPOINT_EVERY:
                    pd.DataFrame(results).to_csv(
                        output_path,
                        index=False,
                    )
                    unsaved_count = 0

            # Final save for the configuration
            pd.DataFrame(results).to_csv(
                output_path,
                index=False,
            )

In [ ]:
generated_filenames = [
    f"{prompt_name}_{model_name}_{corpus_name}.csv"
    for corpus_name, _ in corpus_configs
    for prompt_name, _ in prompt_configs
    for model_name, _ in model_configs
]

print(f"Expected files: {len(generated_filenames)}")

for filename in generated_filenames:
    print(filename)